In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Inference Instructions

This notebook evaluates a trained model by loading an inference CSV and computing the evaluation metrics.

Before running the notebook, update the `CSV_PATH` variable to point to the desired inference CSV generated by the corresponding model.

```python
CSV_PATH = "/path/to/inference_predictions.csv"
```

Below are the default paths for each model.

---

## Relative Ranking Reward

### Run 1

```python
CSV_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/relative_ranking_reward_run1.csv"
```

### Run 2

```python
CSV_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/relative_ranking_reward_run2.csv"
```

---

## Exact Ranking Reward

### Run 1

```python
CSV_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/exact_ranking_reward_run1.csv"
```

### Run 2

```python
CSV_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/exact_ranking_reward_run2.csv"
```

---

## Composite Ranking Reward

### Run 1

```python
CSV_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/composite_ranking_reward_run1.csv"
```

### Run 2

```python
CSV_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/composite_ranking_reward_run2.csv"
```

---

## SFT Model

```python
CSV_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/sft_adapter_predictions.csv"
```

---

## Updating the CSV Path

To evaluate your own inference results, simply replace `CSV_PATH` with the location of your generated inference CSV.

Example:

```python
CSV_PATH = "/content/drive/MyDrive/my_results/model_predictions.csv"
```

The notebook will automatically load the specified prediction file and compute the corresponding evaluation metrics.

FOR RELATIVE RANKING RUN 1

In [ ]:
CSV_PATH = "/content/drive/MyDrive/colm_grpo_llmaj_dataset/test/llama3b/relative_ranking_reward_run1.csv"


In [ ]:
import re
import pandas as pd
from itertools import combinations
from sklearn.metrics import precision_recall_fscore_support



GOLD_COL = "gold_ranking"
PRED_COL = "prediction"
RAW_COL = None

VALID_RANKINGS = [
    "R1>R2>R3",
    "R1>R3>R2",
    "R2>R1>R3",
    "R2>R3>R1",
    "R3>R1>R2",
    "R3>R2>R1",
]

VALID_SET = set(VALID_RANKINGS)

def extract_first_valid_ranking(text):
    if pd.isna(text):
        return None

    text = str(text)

    pattern = r"R[123]\s*>\s*R[123]\s*>\s*R[123]"
    matches = re.findall(pattern, text)

    for m in matches:
        ranking = re.sub(r"\s+", "", m)
        parts = ranking.split(">")
        if ranking in VALID_SET and len(set(parts)) == 3:
            return ranking

    return None


def top1_match(gold, pred):
    if pred is None:
        return 0
    return int(gold.split(">")[0] == pred.split(">")[0])


def pairwise_agreement(gold, pred):
    """
    If prediction is unparsed, returns 0.
    """
    if pred is None:
        return 0.0

    gold_items = gold.split(">")
    pred_items = pred.split(">")

    gold_pos = {x: i for i, x in enumerate(gold_items)}
    pred_pos = {x: i for i, x in enumerate(pred_items)}

    correct = 0
    total = 0

    for a, b in combinations(gold_items, 2):
        total += 1
        if (gold_pos[a] < gold_pos[b]) == (pred_pos[a] < pred_pos[b]):
            correct += 1

    return correct / total

df = pd.read_csv(CSV_PATH)

df["gold_parsed"] = df[GOLD_COL].apply(extract_first_valid_ranking)

if RAW_COL is not None:
    df["pred_parsed"] = df[RAW_COL].apply(extract_first_valid_ranking)
else:
    df["pred_parsed"] = df[PRED_COL].apply(extract_first_valid_ranking)

df = df[df["gold_parsed"].isin(VALID_SET)].copy()

n_total = len(df)
n_parsed = df["pred_parsed"].notna().sum()


parse_rate = n_parsed / n_total

exact_match = (df["gold_parsed"] == df["pred_parsed"]).mean()

top1_accuracy = df.apply(
    lambda row: top1_match(row["gold_parsed"], row["pred_parsed"]),
    axis=1
).mean()

pairwise = df.apply(
    lambda row: pairwise_agreement(row["gold_parsed"], row["pred_parsed"]),
    axis=1
).mean()

parsed_df = df[df["pred_parsed"].notna()].copy()

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    parsed_df["gold_parsed"],
    parsed_df["pred_parsed"],
    labels=VALID_RANKINGS,
    average="macro",
    zero_division=0,
)

metrics = {
    "n_total": n_total,
    "n_parsed": int(n_parsed),
    "parse_rate": round(parse_rate, 6),
    "exact_match": round(exact_match, 6),
    "top1_accuracy": round(top1_accuracy, 6),
    "pairwise_agreement": round(pairwise, 6),
    "macro_precision": round(macro_precision, 6),
    "macro_recall": round(macro_recall, 6),
    "macro_f1": round(macro_f1, 6),
}

metrics